In [ ]:
import numpy as np
import pandas as pd
import GPy
from sklearn.model_selection import train_test_split

In [ ]:
# ===================================
# Configuration
# ===================================

shuffle_data = False
save_data = False
prediction_harmonic_signal = True

mean = 0
stddev = 0  # Add noise if needed

# ===================================
# Load datasets (shared across all cases)
# ===================================

dataset_static = np.load('Dataset\\CFDD\\dataset_CL0_CM0_M_AoA.npy')         # [samples, 1, 4] => M, AoA, CL0, CM0
dataset_linear = np.load('Dataset\\Volterra\\kernels_CL_CM_linear.npy')         # [samples, timesteps, 2]
dataset_nonlinear = np.load('Dataset\\Volterra\\kernels_CL_CM_NL.npy')          # [samples, timesteps, 2]

In [ ]:
# ===================================
# Prepare Input/Output based on case
# ===================================

def prepare_data(case):
    if case == 'CL_linear':
        X = dataset_static[:, 0, [0, 1, 2]]
        Y = dataset_linear[:, :, 0]
    elif case == 'CL_NL':
        X = dataset_static[:, 0, [0, 1, 2]]
        Y = dataset_nonlinear[:, :, 0]
    elif case == 'CM_linear':
        X = dataset_static[:, 0, [0, 1, 3]]
        Y = dataset_linear[:, :, 1]
    elif case == 'CM_NL':
        X = dataset_static[:, 0, [0, 1, 3]]
        Y = dataset_nonlinear[:, :, 1]
    else:
        raise ValueError("Invalid case")

    if stddev > 0:
        noise = np.random.normal(loc=mean, scale=abs(X * stddev), size=X.shape)
        X += noise

    # For NL cases, separate last time step
    Y_last = np.expand_dims(Y[:, -1], axis=1) if 'NL' in case else None
    Y = Y[:, :-1] if 'NL' in case else Y

    return X, Y, Y_last

# ===================================
# Train GP model
# ===================================

def train_gp_model(X_train, Y_train):
    kernel = GPy.kern.RBF(input_dim=X_train.shape[1], ARD=True) + \
             GPy.kern.Matern52(input_dim=X_train.shape[1], ARD=True)
    model = GPy.models.GPRegression(X_train, Y_train, kernel)
    model.optimize(optimizer='lbfgsb', max_iters=1000)
    return model

# ===================================
# Predict harmonic response
# ===================================

def predict_harmonic(model, case, Y_last):
    Mach = 0.70
    AoA_vals = np.array([5.0, 5.0])
    Mach_vals = np.full_like(AoA_vals, Mach)
    CL0_vals = np.full_like(AoA_vals, 0.6153458972)
    CM0_vals = np.full_like(AoA_vals, -0.04452068245)

    if 'CL' in case:
        X_exp = np.stack((Mach_vals, AoA_vals, CL0_vals), axis=1)
    else:
        X_exp = np.stack((Mach_vals, AoA_vals, CM0_vals), axis=1)

    Y_pred_exp, std_exp = model.predict(X_exp)
    if Y_last is not None:
        Y_pred_exp = np.append(Y_pred_exp, Y_last[:2], axis=1)

    np.save(f'Dataset\\Predictions\\kernels_{case}_pred_GP_M070_AoA5.npy', Y_pred_exp)
    np.save(f'Dataset\\Predictions\\variance_kernels_{case}_pred_GP_M070_AoA5.npy', std_exp)


In [ ]:
for case_study in ['CL_linear', 'CL_NL', 'CM_linear', 'CM_NL']:
    print(f"\n=== Processing case: {case_study} ===")

    # Prepare data
    X, Y, Y_last = prepare_data(case_study)
    print(f"Input shape: {X.shape}, Output shape: {Y.shape}")

    # Split dataset
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.42, shuffle=shuffle_data)
    X_val, X_test, Y_val, Y_test = train_test_split(X_test, Y_test, test_size=0.5, shuffle=shuffle_data)

    # Train GP model
    model = train_gp_model(X_train, Y_train)

    # Predict full set
    Y_pred, Y_std = model.predict(X)
    if Y_last is not None:
        Y_pred = np.append(Y_pred, Y_last, axis=1)

    mse = ((Y_pred - np.append(Y, Y_last, axis=1) if Y_last is not None else Y)**2).mean()
    print(f"MSE for {case_study}: {mse:.6f}")

    # Save data
    if save_data:
        np.save(f'Dataset\\Predictions\\kernels_{case_study}_pred_GP.npy', Y_pred)
        np.save(f'Dataset\\Predictions\\variance_kernels_{case_study}_pred_GP.npy', Y_std)

    # Harmonic prediction
    if prediction_harmonic_signal:
        predict_harmonic(model, case_study, Y_last)
